In [2]:
import json
import time
from sentence_transformers import SentenceTransformer, util


INPUT_FILE = "dataset_sample_10k.json"
OUTPUT_FILE = "semantic_relevance_results.json"

# LOAD EMBEDDING MODEL
model = SentenceTransformer("all-MiniLM-L6-v2")


# KEYWORDS NORMALIZATION
def normalize_keywords(keywords):
    if isinstance(keywords, list):
        return [k.lower().strip() for k in keywords if isinstance(k, str)]
    elif isinstance(keywords, str):
        return [keywords.lower().strip()]
    return []


def get_title_text(item):
    return item.get("title", "")


def get_description_text(item):
    return item.get("description", "")


# -------- MAIN --------
if __name__ == "__main__":

    start_time = time.time()

    print("🔄 Loading dataset...\n")
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"Loaded {len(data)} datasets\n")

    # -------- PREPARE TEXT --------
    title_texts = []
    description_texts = []
    keyword_lists = []

    for d in data:
        keywords = normalize_keywords(d.get("keywords", []))
        keyword_lists.append(keywords)

        title_texts.append(get_title_text(d))
        description_texts.append(get_description_text(d))

    # -------- EMBEDDINGS --------
    print("🔄 Computing embeddings...")

    embed_start = time.time()

    title_embeddings = model.encode(
        title_texts,
        convert_to_tensor=True,
        batch_size=64
    )

    description_embeddings = model.encode(
        description_texts,
        convert_to_tensor=True,
        batch_size=64
    )

    embed_end = time.time()

    print(f"✅ Embeddings computed in {embed_end - embed_start:.2f}s\n")

    # -------- SIMILARITY --------
    print("🔄 Computing semantic relevance...\n")

    results = []

    for i in range(len(data)):

        title_emb = title_embeddings[i]
        description_emb = description_embeddings[i]

        keywords = keyword_lists[i]

        if len(keywords) > 0:

            keyword_embs = model.encode(
                keywords,
                convert_to_tensor=True,
                batch_size=32
            )

            # ---- Aggregated similarity (mean pooling) ----
            keyword_emb = keyword_embs.mean(dim=0)

            agg_title_sim = float(
                util.cos_sim(keyword_emb, title_emb)[0][0]
            )

            agg_description_sim = float(
                util.cos_sim(keyword_emb, description_emb)[0][0]
            )

            # ---- Per-keyword similarity ----
            title_sims = util.cos_sim(
                keyword_embs,
                title_emb
            ).cpu().numpy()

            avg_title_sim = float(title_sims.mean())

            description_sims = util.cos_sim(
                keyword_embs,
                description_emb
            ).cpu().numpy()

            avg_description_sim = float(description_sims.mean())

        else:
            agg_title_sim = 0.0
            agg_description_sim = 0.0
            avg_title_sim = 0.0
            avg_description_sim = 0.0

        results.append({
            "dataset_id": data[i].get("dataset_id"),

            "agg_title_similarity": agg_title_sim,
            "agg_description_similarity": agg_description_sim,

            "avg_title_similarity": avg_title_sim,
            "avg_description_similarity": avg_description_sim,

            "num_keywords": len(keywords)
        })

        if (i + 1) % 500 == 0:
            print(f"Processed {i+1}/{len(data)} datasets")

    # -------- SAVE RESULTS --------
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"\n💾 Results saved to {OUTPUT_FILE}")

    end_time = time.time()
    print(f"\n⏱ Total time: {end_time - start_time:.2f}s")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔄 Loading dataset...

Loaded 10000 datasets

🔄 Computing embeddings...
✅ Embeddings computed in 612.72s

🔄 Computing semantic relevance...

Processed 500/10000 datasets
Processed 1000/10000 datasets
Processed 1500/10000 datasets
Processed 2000/10000 datasets
Processed 2500/10000 datasets
Processed 3000/10000 datasets
Processed 3500/10000 datasets
Processed 4000/10000 datasets
Processed 4500/10000 datasets
Processed 5000/10000 datasets
Processed 5500/10000 datasets
Processed 6000/10000 datasets
Processed 6500/10000 datasets
Processed 7000/10000 datasets
Processed 7500/10000 datasets
Processed 8000/10000 datasets
Processed 8500/10000 datasets
Processed 9000/10000 datasets
Processed 9500/10000 datasets
Processed 10000/10000 datasets

💾 Results saved to semantic_relevance_results.json

⏱ Total time: 1843.56s
